# Tema 7 · Entrenar una red en GPU y guardarla en Drive
*Curso «Google Colab» · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** entrenar una red convolucional pequeña, medir qué tan bien funciona y **guardarla en tu Drive** para no
perderla cuando la máquina se borre.

Usamos **Fashion-MNIST**: 70.000 imágenes de 28×28 píxeles de ropa en 10 clases (polera, pantalón, zapatilla…),
publicadas por Zalando Research con licencia **MIT**.

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
# La prueba automática de MiAtlas.cl define MIATLAS_PRUEBA=1 para correr con pocos datos. Tú no necesitas tocarla.
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO, "| prueba rápida:", PRUEBA_RAPIDA)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU: el cuaderno funciona, pero más lento. En Colab: Entorno de ejecución → Cambiar tipo → GPU.")

In [ ]:
from pathlib import Path
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/MiAtlas-Colab")
else:
    CARPETA = Path("MiAtlas-Colab")
CARPETA.mkdir(parents=True, exist_ok=True)
print("Carpeta de trabajo:", CARPETA)

## 1. Los datos

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

transformar = transforms.ToTensor()          # imagen → tensor con valores entre 0 y 1
entrenamiento = datasets.FashionMNIST("datos", train=True, download=True, transform=transformar)
prueba = datasets.FashionMNIST("datos", train=False, download=True, transform=transformar)
if PRUEBA_RAPIDA:
    entrenamiento, prueba = Subset(entrenamiento, range(2000)), Subset(prueba, range(500))
CLASES = ["polera", "pantalón", "suéter", "vestido", "abrigo", "sandalia", "camisa", "zapatilla", "bolso", "botín"]
print(len(entrenamiento), "imágenes para entrenar y", len(prueba), "para probar")

carga_ent = DataLoader(entrenamiento, batch_size=128, shuffle=True)
carga_pru = DataLoader(prueba, batch_size=256)

In [ ]:
import matplotlib.pyplot as plt

fig, ejes = plt.subplots(1, 8, figsize=(12, 2))
for eje, i in zip(ejes, range(8)):
    imagen, etiqueta = entrenamiento[i]
    eje.imshow(imagen.squeeze(), cmap="gray")
    eje.set_title(CLASES[etiqueta], fontsize=9)
    eje.axis("off")
plt.show()

## 2. La red

Dos capas **convolucionales** (detectan bordes y formas) y una capa final que decide entre las 10 clases.

In [ ]:
from torch import nn

class RedRopa(nn.Module):
    def __init__(self):
        super().__init__()
        self.convoluciones = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 28×28 → 14×14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 14×14 → 7×7
        )
        self.clasificador = nn.Sequential(nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        return self.clasificador(self.convoluciones(x))

torch.manual_seed(0)
modelo = RedRopa().to(DISPOSITIVO)      # el modelo también va a la GPU
print(modelo)

## 3. Entrenar

In [ ]:
import time

def exactitud(modelo, cargador):
    modelo.eval()
    correctas = total = 0
    with torch.no_grad():
        for x, y in cargador:
            x, y = x.to(DISPOSITIVO), y.to(DISPOSITIVO)
            correctas += (modelo(x).argmax(1) == y).sum().item()
            total += y.numel()
    return correctas / total

optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
perdida_fn = nn.CrossEntropyLoss()
EPOCAS = 1 if PRUEBA_RAPIDA else 5

for epoca in range(1, EPOCAS + 1):
    modelo.train()
    inicio = time.perf_counter()
    for x, y in carga_ent:
        x, y = x.to(DISPOSITIVO), y.to(DISPOSITIVO)    # cada lote viaja a la GPU
        optimizador.zero_grad()
        perdida = perdida_fn(modelo(x), y)
        perdida.backward()
        optimizador.step()
    print(f"Época {epoca}: pérdida {perdida.item():.3f} · exactitud en prueba {exactitud(modelo, carga_pru):.1%}"
          f" · {time.perf_counter() - inicio:.1f} s")

> **Experimenta:** anota cuántos segundos tarda una época con GPU. Después cambia a CPU (Entorno de ejecución →
> Cambiar tipo → CPU), vuelve a ejecutar todo y compara.

## 4. Guardar en Drive y volver a cargar

Se guarda el **estado** del modelo (sus pesos), no el objeto entero: es más liviano y no depende de la versión de tu
código.

In [ ]:
ruta = CARPETA / "red_ropa.pt"
torch.save(modelo.state_dict(), ruta)
print("Guardado en", ruta, f"({ruta.stat().st_size / 1024:.0f} KB)")

copia = RedRopa().to(DISPOSITIVO)
copia.load_state_dict(torch.load(ruta, map_location=DISPOSITIVO))
print("La copia cargada acierta lo mismo:", f"{exactitud(copia, carga_pru):.1%}")

## Pruébalo tú

Escribe `contar_parametros(modelo)`: cuántos números (pesos) tiene la red en total. Pista: cada `p` de
`modelo.parameters()` tiene `p.numel()`.

In [ ]:
def contar_parametros(modelo):
    # ← tu código aquí
    return None

In [ ]:
# Comprobación
n = contar_parametros(modelo)
if n is None:
    print("Aún no lo resuelves.")
elif n == 421642:
    print(f"✅ ¡Correcto! La red tiene {n:,} parámetros".replace(",", "."))
else:
    print("❌ Obtuviste", n, "— revisa que sumes p.numel() de todos los parámetros")

## Resumen

- El modelo y cada lote de datos van a la GPU con `.to(DISPOSITIVO)`.
- `torch.save(modelo.state_dict(), ruta)` en **Drive** conserva el modelo entrenado.
- `load_state_dict(torch.load(...))` lo recupera en otra sesión.

**Fuentes:** [Fashion-MNIST (Zalando Research, MIT)](https://github.com/zalandoresearch/fashion-mnist) · [PyTorch · guardar y cargar modelos](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html) · [torchvision · FashionMNIST](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html)